# Lab 06 — Development Runner

This notebook is the **only development notebook that owns visible widgets**.

It is **not part of the production Gold Job**.

## Purpose

Use this notebook when you want to manually test one recurring Lab 06 notebook
or run the complete Gold chain without putting parameter widgets into every
processing notebook.

```text
Dev Runner parameters
        ↓
dbutils.notebook.run(...)
        ↓
runtime_config.py
        ↓
processing notebook
```

Production remains:

```text
lab06_gold_job.yml
        ↓
Job parameters
        ↓
runtime_config.py
        ↓
01 → 02 → 03 → 04 → 07
```

## 1. Development parameters

In [0]:
dbutils.widgets.dropdown(
    "task",
    "01_dimensions",
    [
        "01_dimensions",
        "02_fact_encounters",
        "03_fact_conditions",
        "04_aggregations",
        "07_validation",
        "FULL_GOLD_CHAIN",
    ],
    "01 Task",
)

dbutils.widgets.text(
    "catalog",
    "dbr_dev",
    "02 Catalog",
)

dbutils.widgets.text(
    "schema",
    "parvinbadalov",
    "03 Schema",
)

dbutils.widgets.text(
    "volume_name",
    "lab06_gold_analytics",
    "04 External volume",
)

dbutils.widgets.dropdown(
    "run_validation",
    "true",
    ["true", "false"],
    "05 Run validation",
)

dbutils.widgets.text(
    "date_start",
    "1900-01-01",
    "06 dim_date start",
)

dbutils.widgets.text(
    "date_end",
    "2035-12-31",
    "07 dim_date end",
)

dbutils.widgets.dropdown(
    "rebuild_dim_date",
    "false",
    ["false", "true"],
    "08 Rebuild dim_date",
)

task = dbutils.widgets.get("task")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume_name = dbutils.widgets.get("volume_name")
run_validation = dbutils.widgets.get("run_validation")
date_start = dbutils.widgets.get("date_start")
date_end = dbutils.widgets.get("date_end")
rebuild_dim_date = dbutils.widgets.get("rebuild_dim_date")

print(f"Task             : {task}")
print(f"Catalog          : {catalog}")
print(f"Schema           : {schema}")
print(f"Volume           : {volume_name}")
print(f"Run validation   : {run_validation}")
print(f"Date range       : {date_start} -> {date_end}")
print(f"Rebuild dim_date : {rebuild_dim_date}")

## 2. Runner configuration

In [0]:
NOTEBOOKS = {
    "01_dimensions": "./lab06_01_dimensions",
    "02_fact_encounters": "./lab06_02_fact_encounters",
    "03_fact_conditions": "./lab06_03_fact_conditions",
    "04_aggregations": "./lab06_04_aggregations",
    "07_validation": "./lab06_07_validation",
}

COMMON_ARGS = {
    "catalog": catalog,
    "schema": schema,
    "volume_name": volume_name,
}

DIMENSION_ARGS = {
    **COMMON_ARGS,
    "date_start": date_start,
    "date_end": date_end,
    "rebuild_dim_date": rebuild_dim_date,
}

VALIDATION_ARGS = {
    **COMMON_ARGS,
    "run_validation": run_validation,
}

TASK_ARGS = {
    "01_dimensions": DIMENSION_ARGS,
    "02_fact_encounters": VALIDATION_ARGS,
    "03_fact_conditions": VALIDATION_ARGS,
    "04_aggregations": VALIDATION_ARGS,
    "07_validation": VALIDATION_ARGS,
}

## 3. Execute selected task

In [0]:
def run_lab06_notebook(task_key: str) -> str:
    notebook_path = NOTEBOOKS[task_key]
    arguments = TASK_ARGS[task_key]

    print("")
    print("=" * 70)
    print(f"START: {task_key}")
    print(f"Path : {notebook_path}")
    print("=" * 70)

    result = dbutils.notebook.run(
        notebook_path,
        timeout_seconds=0,
        arguments=arguments,
    )

    print("")
    print(f"SUCCESS: {task_key}")

    if result:
        print(f"Return value: {result}")

    return result


if task == "FULL_GOLD_CHAIN":
    execution_order = [
        "01_dimensions",
        "02_fact_encounters",
        "03_fact_conditions",
        "04_aggregations",
        "07_validation",
    ]

    completed = []

    for task_key in execution_order:
        run_lab06_notebook(task_key)
        completed.append(task_key)

    print("")
    print("LAB 06 — DEVELOPMENT FULL GOLD CHAIN COMPLETE")
    print("Completed:")
    for task_key in completed:
        print(f"  - {task_key}")

else:
    run_lab06_notebook(task)

    print("")
    print("LAB 06 — DEVELOPMENT TASK COMPLETE")
    print(f"Completed: {task}")

## How to use

For a quick test:

1. Choose a value in **Task**.
2. Keep the default development parameters unless you intentionally need a change.
3. Click **Run all**.

Recommended first test after the runtime-config refactor:

```text
Task = 01_dimensions
```

If that succeeds, test:

```text
Task = FULL_GOLD_CHAIN
```

This runner is a development convenience only. Do **not** add it to
`resources/lab06_gold_job.yml`.